# Project: Training an AI to Recognize Images
In this notebook, we are going to teach a computer how to tell the difference between two things (like Ants vs. Bees).

We will use a technique called **Transfer Learning**.
* **Imagine:** We are hiring a genius student who already knows what "lines," "circles," and "shapes" look like.
* **Our Job:** We just need to teach them to use those skills to recognize our specific images.

In [1]:
# 1_train_model.ipynb

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, random_split
import os
import time
import zipfile
import urllib.request

# --- CONFIGURATION (Hyperparameters) ---
BATCH_SIZE = 4       # How many images the model sees at once
LEARNING_RATE = 0.001 # How fast the model learns
EPOCHS = 5           # How many times to loop through the entire dataset
DATA_DIR = './data/hymenoptera_data' # Folder where images are stored
MODEL_SAVE_PATH = 'best_mobilenet.pth'

# Detect if we have a GPU (NVIDIA), MPS (Mac), or CPU
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Training on device: {device}")



Training on device: cuda


## Step 1: Get the Data
Computers learn from examples. We need to download a folder of images.
Inside the main folder, we must have sub-folders for each category.

* `data/train/ants` -> contains photos of ants
* `data/train/bees` -> contains photos of bees

*Note: If you want to use your own data later, just replace the folder structure!*

In [2]:
# --- STEP 1: DOWNLOAD & PREPARE DATA ---
# For this tutorial, we will download a sample dataset (Ants vs Bees)
# It is already organized as: folder_name = category_name
def download_data():
    if not os.path.exists(DATA_DIR):
        print("Downloading dataset...")
        os.makedirs('./data', exist_ok=True)
        url = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"
        zip_path = "./data/hymenoptera_data.zip"
        urllib.request.urlretrieve(url, zip_path)
        
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall("./data")
        print("Download complete.")
    else:
        print("Dataset already exists.")

download_data()


Download complete.


## Step 2: Preparing the "Flashcards" (DataLoaders)
Before showing images to the AI, we need to:
1.  **Resize** them so they are all the same shape (224x224 pixels).
2.  **Turn them into Numbers** (Tensors) because computers can't "see" pixels, they only understand math.
3.  **Shuffle** them so the model doesn't memorize the order.

In [3]:

# Define how we modify images before giving them to the model
# Models expect a standard size (224x224) and numerical normalization
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)), # Resize all images to 224x224
    transforms.ToTensor(),         # Convert image to math numbers (Tensor)
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # Standard colors
])

# Load data from folders
# The 'train' folder in the dataset contains the images for learning
full_dataset = datasets.ImageFolder(root=os.path.join(DATA_DIR, 'train'), transform=data_transforms)

# Split into Train (80%) and Test (20%)
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

# Create DataLoaders (The tools that feed data to the model)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

class_names = full_dataset.classes
print(f"Classes found: {class_names}")


Classes found: ['ants', 'bees']


## Step 3: The Brain (MobileNet)
We are downloading **MobileNet**. This is a famous AI brain created by Google.
* It has already looked at millions of images from the internet.
* **The Hack:** We are going to cut off the very last layer (which recognizes 1000 things) and replace it with a new layer that only recognizes our **2 things** (Ants and Bees).

In [4]:

# We use a pre-trained model (learned on ImageNet) to speed up teaching
model = models.mobilenet_v2(weights='DEFAULT')

# MobileNet is built for 1000 classes. We need to change the last layer to match our classes.
# model.classifier[1] is the final output layer of MobileNetV2
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, len(class_names))

model = model.to(device) # Move model to GPU/CPU

# Define Loss function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9)

# --- STEP 3: TRAIN & EVALUATE ---
best_acc = 0.0



## Step 4: Training Loop
This is where the learning happens. For every "Epoch" (loop):
1.  **Train:** The model guesses, we tell it the answer, and it updates its math.
2.  **Test:** We pause and give it a quiz on images it has never seen before.
3.  **Save:** If it gets a better score on the quiz, we save that version of the brain to a file.

In [5]:
print("\nStarting Training...")
for epoch in range(EPOCHS):
    print(f'Epoch {epoch+1}/{EPOCHS}')
    print('-' * 10)

    # --- Training Phase ---
    model.train() # Set model to training mode
    running_loss = 0.0
    running_corrects = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass (predict)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        # Backward pass (learn)
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = running_corrects.double() / len(train_dataset)

    print(f'Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

    # --- Testing/Validation Phase ---
    model.eval() # Set model to evaluation mode
    test_corrects = 0
    
    # We don't need gradients for testing (saves memory)
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            test_corrects += torch.sum(preds == labels.data)

    test_acc = test_corrects.double() / len(test_dataset)
    print(f'Test Acc: {test_acc:.4f}')

    # Save the model if it's the best one so far
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(">> Model improved. Saved.")

print(f'\nTraining complete. Best Test Accuracy: {best_acc:.4f}')
print(f'Model saved to: {MODEL_SAVE_PATH}')


Starting Training...
Epoch 1/5
----------
Train Loss: 0.6636 Acc: 0.5693
Test Acc: 0.8431
>> Model improved. Saved.
Epoch 2/5
----------
Train Loss: 0.4737 Acc: 0.8168
Test Acc: 0.8627
>> Model improved. Saved.
Epoch 3/5
----------
Train Loss: 0.3943 Acc: 0.8366
Test Acc: 0.8824
>> Model improved. Saved.
Epoch 4/5
----------
Train Loss: 0.3459 Acc: 0.8515
Test Acc: 0.9020
>> Model improved. Saved.
Epoch 5/5
----------
Train Loss: 0.2716 Acc: 0.8861
Test Acc: 0.8627

Training complete. Best Test Accuracy: 0.9020
Model saved to: best_mobilenet.pth


In [ ]:
#jupyter notebook --ip=0.0.0.0 --port=8888 --no-browser